# 06 — CatBoost Propensity Experiments

## Objective
The logistic-regression experiments in notebook 05 showed almost no ranking signal (train/validation ROC-AUC ≈ 0.5). This notebook changes the **model family**, not just a hyperparameter.

We test CatBoost because it can learn non-linear interactions among customer, product, channel and campaign context.

**Evaluation rule**
- Train on `train`.
- Compare candidates on `validation`.
- **Do not touch `test` in this notebook.**
- Only after a candidate shows meaningful validation lift do we run a final holdout evaluation.


## 1. Install dependencies

In [12]:
!pip -q install catboost pyarrow scikit-learn joblib

## 2. Mount Drive and import reusable project code

In [13]:
from pathlib import Path
import sys
import json
import joblib
import pandas as pd
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

PROJECT_DIR = Path('/content/drive/MyDrive/03 Projects/Factored Hackathon')
SRC_DIR = PROJECT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from propensity.config import (
    RANDOM_STATE,
    TARGET,
    DATASET_PATH,
    ARTIFACT_DIR,
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
)
from propensity.data import load_dataset, temporal_split, make_xy
from propensity.features import add_propensity_interactions, INTERACTION_FEATURES
from propensity.catboost_modeling import (
    CatBoostConfig,
    train_catboost,
    prepare_catboost_frame,
)
from propensity.evaluation import evaluate_model


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Load data and add pre-send interaction features

In [14]:
df = load_dataset(DATASET_PATH)
df = add_propensity_interactions(df)

CATEGORICAL_FEATURES_CB = CATEGORICAL_FEATURES + INTERACTION_FEATURES
MODEL_FEATURES_CB = NUMERIC_FEATURES + CATEGORICAL_FEATURES_CB

missing = [c for c in MODEL_FEATURES_CB + [TARGET] if c not in df.columns]
assert not missing, f'Missing columns: {missing}'

split = temporal_split(df)
X_train, y_train = make_xy(split.train_df, MODEL_FEATURES_CB, TARGET)
X_val, y_val = make_xy(split.validation_df, MODEL_FEATURES_CB, TARGET)

print('Train rows:', f'{len(X_train):,}', 'conversion rate:', f'{y_train.mean():.4%}')
print('Validation rows:', f'{len(X_val):,}', 'conversion rate:', f'{y_val.mean():.4%}')
print('Model features:', len(MODEL_FEATURES_CB))
print('New interactions:', INTERACTION_FEATURES)


Train rows: 849,821 conversion rate: 0.7324%
Validation rows: 182,104 conversion rate: 0.7457%
Model features: 25
New interactions: ['country_product_channel', 'segment_product', 'campaign_objective_product']


## 4. Define CatBoost experiments

Two controlled variants:
1. **unweighted** — preferred for probability quality if ranking works;
2. **balanced** — checks whether emphasizing the rare positive class improves ranking.

We are deliberately not launching a huge hyperparameter search yet.


In [15]:
experiments = [
    CatBoostConfig(
        name='catboost_interactions_unweighted',
        iterations=500,
        depth=7,
        learning_rate=0.05,
        l2_leaf_reg=5.0,
        auto_class_weights=None,
        random_state=RANDOM_STATE,
        early_stopping_rounds=75,
    ),
    CatBoostConfig(
        name='catboost_interactions_balanced',
        iterations=500,
        depth=7,
        learning_rate=0.05,
        l2_leaf_reg=5.0,
        auto_class_weights='Balanced',
        random_state=RANDOM_STATE,
        early_stopping_rounds=75,
    ),
]

RUN_ID = pd.Timestamp.now(tz='America/Lima').strftime('%Y%m%d_%H%M%S')
RUN_DIR = Path(ARTIFACT_DIR) / 'catboost_experiments' / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Run:', RUN_ID)
print('Artifacts:', RUN_DIR)


Run: 20261004_222052
Artifacts: /content/drive/MyDrive/03 Projects/Factored Hackathon/artifacts/propensity/catboost_experiments/20261004_222052


## 5. Train and compare on train + validation only

In [16]:
trained_models = {}
results = {}
rows = []

# evaluate_model calls predict_proba. CatBoost needs categorical columns prepared
# the same way used during fit.
X_train_eval = prepare_catboost_frame(X_train, CATEGORICAL_FEATURES_CB)
X_val_eval = prepare_catboost_frame(X_val, CATEGORICAL_FEATURES_CB)

for config in experiments:
    print('\n' + '=' * 90)
    print('Experiment:', config.name)

    model = train_catboost(
        X_train,
        y_train,
        X_val,
        y_val,
        CATEGORICAL_FEATURES_CB,
        config,
    )

    train_result = evaluate_model(
        model, X_train_eval, y_train, name=f'{config.name} — TRAIN'
    )
    val_result = evaluate_model(
        model, X_val_eval, y_val, name=f'{config.name} — VALIDATION'
    )

    trained_models[config.name] = model
    results[config.name] = {'train': train_result, 'validation': val_result}

    experiment_dir = RUN_DIR / config.name
    experiment_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, experiment_dir / 'model.joblib')
    train_result['deciles'].to_csv(experiment_dir / 'train_deciles.csv')
    val_result['deciles'].to_csv(experiment_dir / 'validation_deciles.csv')

    metadata = {
        'run_id': RUN_ID,
        'experiment': config.as_dict(),
        'numeric_features': NUMERIC_FEATURES,
        'categorical_features': CATEGORICAL_FEATURES_CB,
        'interaction_features': INTERACTION_FEATURES,
        'model_features': MODEL_FEATURES_CB,
        'train_metrics': {
            'pr_auc': train_result['pr_auc'],
            'roc_auc': train_result['roc_auc'],
            'base_conversion_rate': train_result['base_conversion_rate'],
        },
        'validation_metrics': {
            'pr_auc': val_result['pr_auc'],
            'roc_auc': val_result['roc_auc'],
            'base_conversion_rate': val_result['base_conversion_rate'],
            'best_f1': val_result['best_f1'],
            'best_f1_threshold': val_result['best_f1_threshold'],
        },
    }
    (experiment_dir / 'metadata.json').write_text(
        json.dumps(metadata, indent=2), encoding='utf-8'
    )

    rows.append({
        'experiment': config.name,
        'best_iteration': model.get_best_iteration(),
        'train_pr_auc': train_result['pr_auc'],
        'train_roc_auc': train_result['roc_auc'],
        'val_pr_auc': val_result['pr_auc'],
        'val_roc_auc': val_result['roc_auc'],
        'val_pr_auc_lift_vs_base': (
            val_result['pr_auc'] / val_result['base_conversion_rate']
            if val_result['base_conversion_rate'] > 0 else float('nan')
        ),
        'val_best_f1': val_result['best_f1'],
        'val_threshold': val_result['best_f1_threshold'],
    })

comparison = (
    pd.DataFrame(rows)
    .sort_values(['val_pr_auc', 'val_roc_auc'], ascending=False)
    .reset_index(drop=True)
)
comparison.to_csv(RUN_DIR / 'comparison.csv', index=False)
display(comparison)
print('Saved comparison:', RUN_DIR / 'comparison.csv')



Experiment: catboost_interactions_unweighted
0:	test: 0.5051393	best: 0.5051393 (0)	total: 6.14s	remaining: 51m 6s
50:	test: 0.5445629	best: 0.5445629 (48)	total: 2m 8s	remaining: 18m 51s
100:	test: 0.5504007	best: 0.5504116 (95)	total: 3m 58s	remaining: 15m 42s
150:	test: 0.5497089	best: 0.5504713 (143)	total: 6m 44s	remaining: 15m 34s
200:	test: 0.5519000	best: 0.5521349 (197)	total: 9m 23s	remaining: 13m 58s
250:	test: 0.5506674	best: 0.5521349 (197)	total: 12m 27s	remaining: 12m 21s
Stopped by overfitting detector  (75 iterations wait)

bestTest = 0.5521348709
bestIteration = 197

Shrink model to first 198 iterations.
catboost_interactions_unweighted — TRAIN
  PR-AUC / Average Precision: 0.011235
  ROC-AUC:                    0.594682
  Base conversion rate:       0.007324
catboost_interactions_unweighted — VALIDATION
  PR-AUC / Average Precision: 0.008918
  ROC-AUC:                    0.552135
  Base conversion rate:       0.007457

Experiment: catboost_interactions_balanced
0:	t

,experiment,best_iteration,train_pr_auc,train_roc_auc,val_pr_auc,val_roc_auc,val_pr_auc_lift_vs_base,val_best_f1,val_threshold
0,catboost_interactions_unweighted,197,0.011235,0.594682,0.008918,0.552135,1.195911,0.019576,0.009259
1,catboost_interactions_balanced,99,0.010691,0.600146,0.008907,0.550683,1.194340,0.019315,0.549301


Saved comparison: /content/drive/MyDrive/03 Projects/Factored Hackathon/artifacts/propensity/catboost_experiments/20261004_222052/comparison.csv


## 6. Inspect the best validation candidate

In [17]:
selected_name = comparison.loc[0, 'experiment']
selected_val = results[selected_name]['validation']

print('Selected on validation:', selected_name)
print('Validation PR-AUC:', selected_val['pr_auc'])
print('Validation ROC-AUC:', selected_val['roc_auc'])
print('Base conversion rate:', selected_val['base_conversion_rate'])
print('PR-AUC lift vs base:', selected_val['pr_auc'] / selected_val['base_conversion_rate'])
print('\nValidation deciles:')
display(selected_val['deciles'])


Selected on validation: catboost_interactions_unweighted
Validation PR-AUC: 0.008918236133160954
Validation ROC-AUC: 0.5521348708503411
Base conversion rate: 0.007457277160304002
PR-AUC lift vs base: 1.1959105101569532

Validation deciles:


,rows,conversions,observed_conversion_rate,avg_propensity,lift_vs_random
decile,,,,,
10,18211,183,0.010049,0.009408,1.347526
9,18210,157,0.008622,0.008977,1.156137
8,18210,172,0.009445,0.008684,1.266596
7,18211,150,0.008237,0.008285,1.104529
6,18210,123,0.006755,0.007848,0.905764
5,18210,126,0.006919,0.006938,0.927855
4,18211,124,0.006809,0.006071,0.913077
3,18210,101,0.005546,0.005893,0.743757
2,18210,108,0.005931,0.005745,0.795305


## 7. Automatic diagnostic — what should we do next?

In [18]:
base = selected_val['base_conversion_rate']
pr_lift = selected_val['pr_auc'] / base if base > 0 else 0
roc = selected_val['roc_auc']

print('Best validation ROC-AUC:', round(roc, 4))
print('Best PR-AUC lift vs base:', round(pr_lift, 3), 'x')

if roc >= 0.60 and pr_lift >= 1.5:
    print('NEXT: Strong enough to proceed to calibration + one-time final holdout evaluation.')
elif roc >= 0.55 or pr_lift >= 1.25:
    print('NEXT: There is useful signal. Inspect decile lift, then do a small targeted tuning round.')
else:
    print('NEXT: Signal is still weak. Do NOT burn time on broad hyperparameter tuning.')
    print('Add lagged customer behavior features (prior sends/opens/clicks/conversions) or use a segment-level NBA fallback.')

print('\nImportant: test was intentionally NOT evaluated in this notebook.')


Best validation ROC-AUC: 0.5521
Best PR-AUC lift vs base: 1.196 x
NEXT: There is useful signal. Inspect decile lift, then do a small targeted tuning round.

Important: test was intentionally NOT evaluated in this notebook.


## Next step

Send ChatGPT the output from **cells 5–7** (or just say the notebook finished; it can read the saved notebook/artifacts from Drive).

Decision path:
- useful validation signal → targeted tuning / calibration → final holdout → NBA;
- weak signal → add leakage-safe historical behavior features;
- still weak after behavior features → segment-level historical policy as a transparent fallback.
